# 07 Leak check, step 1: cost timing and the UC3M plate-text chance table (M1b-7)
Runs on Colab (Drive mounted). Plan: PLAN.md (M1b-7) and the decisions of 2026-11-14.

**This notebook only measures.** It times ORB for checks 1 (US test vs CCPD train), 3 (UC3M test vs UC3M train) and 4 (CCPD held-out vs CCPD train), and compares the plate text of the authors' UC3M test plates with their train plates. It removes nothing, sets no threshold and starts no long run. Random pairs here are used for **cost only**, never for a threshold.

**Output rules:** no images and no plate text are printed (counts and timings only), so no cell needs the `remove_output` tag. Restart & Run All before committing.

In [1]:
import csv, hashlib, io, os, pickle, random, shutil, tarfile, time
from collections import Counter
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
from google.colab import drive

In [2]:
# Parameters
SEED = 0
ORB_FEATURES, ORB_RATIO, ORB_MIN_GOOD = 500, 0.75, 8   # same as notebooks 03 and 04
N_IMG = 50          # images timed for the UC3M decode/describe steps (32 MP files)
N_IMG_SMALL = 200   # images timed for the CCPD / US describe steps
N_PAIRS = 200       # random pairs timed per combination (cost only)
N_COPY = 20         # image-vs-altered-copy pairs timed per combination (these reach RANSAC)
UC3M_SIZES = (1024, 1536)   # candidate long sides for UC3M; the choice is made later from recall on a positive control
DRIVE = '/content/drive/MyDrive/alpr'
ARCHIVES = f'{DRIVE}/archives'
LOCAL = '/content/tars'
TARS = {   # name -> (file, MD5 recorded in PLAN.md)
    'ccpd': ('ccpd_yolo.tar', 'bd44986acd6691a4f90fbea1e6017682'),
    'us': ('us_yolo.tar', '1ce491ae6a7accefdb0b9156a30f2acc'),
    'uc3m': ('uc3m_yolo.tar', 'e3ef306731f33095bac983338c6adb04'),
}
LABELS_CSV = f'{ARCHIVES}/uc3m_labels.csv'   # contains partial plate text: Drive only, never printed

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
%%bash
set -e
cd /content
if [ -d vehicle-plate-recognition ]; then git -C vehicle-plate-recognition pull --ff-only; else git clone https://github.com/medrees-1000/vehicle-plate-recognition.git; fi
pip install -q uv  # approved raw pip: only to bootstrap uv on Colab
uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition

Already up to date.


In [5]:
# setup: copy the three tars to the VM disk, check MD5, list members (names stay in memory, nothing is printed)
from alpr import ccpd, uc3m   # after the install cell (ImportError on the first run? restart the kernel once)
Path(LOCAL).mkdir(exist_ok=True)
MEM = {}   # name -> {split: [member names]}
for name, (fn, md5) in TARS.items():
    src, dst = Path(ARCHIVES, fn), Path(LOCAL, fn)
    assert src.exists(), f'{fn} not found in {ARCHIVES}'
    if not dst.exists():
        t0 = time.time(); shutil.copyfile(src, dst); print(f'{name}: copied in {time.time() - t0:.0f} s')
    got = ccpd.file_md5(dst)
    assert got == md5, f'{name}: MD5 {got} differs from PLAN.md ({md5})'
    with tarfile.open(dst) as t:
        MEM[name] = {}
        for m in t.getmembers():
            if m.isfile() and m.name.startswith('images/'):
                MEM[name].setdefault(m.name.split('/')[1], []).append(m.name)
    print(name, 'MD5 ok;', {s: len(v) for s, v in MEM[name].items()}, 'images')
assert len(MEM['ccpd']['train']) == 16556 and len(MEM['ccpd']['heldout']) == 300 and len(MEM['us']['test']) == 105
assert len(MEM['uc3m']['train']) == 1580 and len(MEM['uc3m']['test']) == 395

ccpd: copied in 25 s
ccpd MD5 ok; {'train': 16556, 'val': 997, 'heldout': 300} images
us: copied in 1 s
us MD5 ok; {'test': 105, 'val': 92, 'train': 3373} images
uc3m: copied in 33 s
uc3m MD5 ok; {'test': 395, 'train': 1580} images


In [6]:
# helpers (ORB as in notebooks 03/04; timed pieces are split so the RANSAC share is visible)
TAR = {n: tarfile.open(Path(LOCAL, fn)) for n, (fn, _) in TARS.items()}
def read(name, member): return TAR[name].extractfile(member).read()

def load_gray(data, long_side=None):
    im = Image.open(io.BytesIO(data))
    if long_side:
        im.draft('L', (long_side, long_side))   # JPEG DCT scaling: decodes at 1/2, 1/4 or 1/8 size when that is still >= the request
    g = np.asarray(im.convert('L'))
    if long_side and max(g.shape) > long_side:
        s = long_side / max(g.shape)
        g = cv2.resize(g, (round(g.shape[1] * s), round(g.shape[0] * s)), interpolation=cv2.INTER_AREA)
    return g

ORB = cv2.ORB_create(nfeatures=ORB_FEATURES)
def describe(g):
    kp, des = ORB.detectAndCompute(g, None)
    return (np.float32([k.pt for k in kp]), des) if des is not None and len(kp) else (np.zeros((0, 2), np.float32), None)

BF = cv2.BFMatcher(cv2.NORM_HAMMING)
def score_timed(a, b):
    """(inliers, seconds of knn+ratio, seconds of RANSAC or None when fewer than ORB_MIN_GOOD good matches)."""
    (pa, da), (pb, db) = a, b
    if da is None or db is None or len(da) < 2 or len(db) < 2:
        return 0, 0.0, None
    t0 = time.perf_counter()
    good = [m[0] for m in BF.knnMatch(da, db, k=2) if len(m) == 2 and m[0].distance < ORB_RATIO * m[1].distance]
    t1 = time.perf_counter()
    if len(good) < ORB_MIN_GOOD:
        return 0, t1 - t0, None
    M, inl = cv2.estimateAffinePartial2D(pa[[m.queryIdx for m in good]], pb[[m.trainIdx for m in good]], method=cv2.RANSAC, ransacReprojThreshold=4.0)
    return (int(inl.sum()) if M is not None else 0), t1 - t0, time.perf_counter() - t1

def altered(g, rng):   # synthetic altered copy: rotate up to 10 deg, crop up to 15%, noise
    h, w = g.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-10, 10), 1.0)
    r = cv2.warpAffine(g, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
    c = rng.uniform(0, 0.15); y, x = int(h * c / 2), int(w * c / 2)
    r = r[y:h - y, x:w - x].astype(np.float32) + np.random.RandomState(rng.randrange(2**31)).normal(0, 6, (h - 2 * y, w - 2 * x))
    return np.clip(r, 0, 255).astype(np.uint8)

def ms(x): return f'{x * 1000:.2f} ms'

In [7]:
# 1. image cost: read, decode(+resize), describe (50 UC3M images, 200 of the others; cost only)
rng = random.Random(SEED)
SAMPLE = {}   # (name, split, size) -> (list of gray arrays, descriptors)
def time_images(name, split, n, size):
    members = rng.sample(MEM[name][split], n)
    t0 = time.perf_counter(); raw = [read(name, m) for m in members]; t_read = (time.perf_counter() - t0) / n
    t0 = time.perf_counter(); grays = [load_gray(d, size) for d in raw]; t_dec = (time.perf_counter() - t0) / n
    t0 = time.perf_counter(); des = [describe(g) for g in grays]; t_des = (time.perf_counter() - t0) / n
    SAMPLE[(name, split, size)] = (grays, des)
    kp = np.mean([len(d[0]) for d in des])
    print(f'{name}/{split} size={size}: n={n} read {ms(t_read)} | decode+resize {ms(t_dec)} | describe {ms(t_des)} | mean keypoints {kp:.0f} | mean image {np.mean([g.shape[1] for g in grays]):.0f}x{np.mean([g.shape[0] for g in grays]):.0f}')
    return t_read + t_dec + t_des
PER_IMG = {}
for size in UC3M_SIZES:
    for split in ('train', 'test'):
        PER_IMG[('uc3m', split, size)] = time_images('uc3m', split, N_IMG, size)
for name, split in (('ccpd', 'train'), ('ccpd', 'heldout'), ('us', 'test')):
    PER_IMG[(name, split, None)] = time_images(name, split, min(N_IMG_SMALL, len(MEM[name][split])), None)

uc3m/train size=1024: n=50 read 12.12 ms | decode+resize 74.66 ms | describe 17.84 ms | mean keypoints 492 | mean image 1001x740
uc3m/test size=1024: n=50 read 22.63 ms | decode+resize 97.32 ms | describe 12.74 ms | mean keypoints 498 | mean image 1024x724
uc3m/train size=1536: n=50 read 9.47 ms | decode+resize 98.31 ms | describe 26.53 ms | mean keypoints 495 | mean image 1519x1106
uc3m/test size=1536: n=50 read 14.68 ms | decode+resize 131.70 ms | describe 25.67 ms | mean keypoints 498 | mean image 1536x1093
ccpd/train size=None: n=200 read 9.29 ms | decode+resize 4.83 ms | describe 14.77 ms | mean keypoints 482 | mean image 720x1160
ccpd/heldout size=None: n=200 read 1.86 ms | decode+resize 6.00 ms | describe 22.88 ms | mean keypoints 489 | mean image 720x1160
us/test size=None: n=105 read 7.15 ms | decode+resize 1.48 ms | describe 6.40 ms | mean keypoints 498 | mean image 416x416


In [8]:
# 2. pair cost: random pairs (cost only) and image-vs-altered-copy pairs (these reach RANSAC)
def time_pairs(label, qa, ta):
    (_, qd), (_, td) = SAMPLE[qa], SAMPLE[ta]
    qg = SAMPLE[qa][0]
    r = random.Random(SEED + 1)
    res = [score_timed(qd[r.randrange(len(qd))], td[r.randrange(len(td))]) for _ in range(N_PAIRS)]
    knn = np.mean([x[1] for x in res]); ran = [x[2] for x in res if x[2] is not None]
    cp = []
    for i in range(min(N_COPY, len(qg))):
        cp.append(score_timed(qd[i], describe(altered(qg[i], r))))
    cp_ran = [x[2] for x in cp if x[2] is not None]
    ransac = np.mean(ran + cp_ran) if ran + cp_ran else None
    print(f'{label}: knn+ratio {ms(knn)}/pair | random pairs that reached RANSAC: {len(ran)}/{N_PAIRS} | altered copies that reached RANSAC: {len(cp_ran)}/{len(cp)} (inliers {sorted(x[0] for x in cp)[:3]}..) | RANSAC {ms(ransac) if ransac is not None else "not measured"}/call')
    return knn, (len(ran) / N_PAIRS), ransac
PAIR = {}
PAIR['1'] = time_pairs('check 1  US test x CCPD train', ('us', 'test', None), ('ccpd', 'train', None))
PAIR['4'] = time_pairs('check 4  CCPD held-out x CCPD train', ('ccpd', 'heldout', None), ('ccpd', 'train', None))
for size in UC3M_SIZES:
    PAIR[f'3@{size}'] = time_pairs(f'check 3  UC3M test x train @ {size}', ('uc3m', 'test', size), ('uc3m', 'train', size))

check 1  US test x CCPD train: knn+ratio 4.03 ms/pair | random pairs that reached RANSAC: 19/200 | altered copies that reached RANSAC: 20/20 (inliers [224, 252, 259]..) | RANSAC 0.88 ms/call
check 4  CCPD held-out x CCPD train: knn+ratio 3.88 ms/pair | random pairs that reached RANSAC: 51/200 | altered copies that reached RANSAC: 20/20 (inliers [214, 214, 218]..) | RANSAC 0.39 ms/call
check 3  UC3M test x train @ 1024: knn+ratio 4.11 ms/pair | random pairs that reached RANSAC: 30/200 | altered copies that reached RANSAC: 20/20 (inliers [239, 247, 261]..) | RANSAC 0.42 ms/call
check 3  UC3M test x train @ 1536: knn+ratio 4.12 ms/pair | random pairs that reached RANSAC: 32/200 | altered copies that reached RANSAC: 20/20 (inliers [153, 257, 271]..) | RANSAC 0.44 ms/call


In [9]:
# 3. cost table: pairs x measured per-pair time (single process). This is arithmetic on the sample above, not a full-run measurement
CHECKS = {'1': (105, 16556, ('us', 'test', None), ('ccpd', 'train', None)),
          '4': (300, 16556, ('ccpd', 'heldout', None), ('ccpd', 'train', None)),
          **{f'3@{s}': (395, 1580, ('uc3m', 'test', s), ('uc3m', 'train', s)) for s in UC3M_SIZES}}
print(f'cores: os.cpu_count() = {os.cpu_count()}, cv2.getNumThreads() = {cv2.getNumThreads()} (whether knnMatch uses them was not measured)')
for k, (nq, nt, q, t) in CHECKS.items():
    knn, frac, ransac = PAIR[k]
    per = knn + frac * (ransac or 0)
    n = nq * nt
    desc = nq * PER_IMG[q] + nt * PER_IMG[t]   # describe both sides once (a cache on Drive makes this one-off)
    print(f'check {k}: {n:,} pairs x {ms(per)} = {n * per / 3600:.2f} h (random-pair share of RANSAC: {frac:.3f}; RANSAC on true copies not included) | one-off image reading + describe: {desc / 60:.1f} min')
print('Not measured: RANSAC cost on the real tail of the pairs, Drive read speed during the full run, multi-process speed-up.')

cores: os.cpu_count() = 2, cv2.getNumThreads() = 2 (whether knnMatch uses them was not measured)
check 1: 1,738,380 pairs x 4.11 ms = 1.98 h (random-pair share of RANSAC: 0.095; RANSAC on true copies not included) | one-off image reading + describe: 8.0 min
check 4: 4,966,800 pairs x 3.98 ms = 5.49 h (random-pair share of RANSAC: 0.255; RANSAC on true copies not included) | one-off image reading + describe: 8.1 min
check 3@1024: 624,100 pairs x 4.17 ms = 0.72 h (random-pair share of RANSAC: 0.150; RANSAC on true copies not included) | one-off image reading + describe: 3.6 min
check 3@1536: 624,100 pairs x 4.19 ms = 0.73 h (random-pair share of RANSAC: 0.160; RANSAC on true copies not included) | one-off image reading + describe: 4.7 min
Not measured: RANSAC cost on the real tail of the pairs, Drive read speed during the full run, multi-process speed-up.


In [10]:
# 4. UC3M plate text: authors' test plates vs train plates, counts only (no text printed)
rows = list(csv.DictReader(open(LABELS_CSV, newline='', encoding='utf-8')))
test_lp = [r['lp_id'] for r in rows if r['split'] == 'test']
train_lp = [r['lp_id'] for r in rows if r['split'] == 'train']
cnt = uc3m.plate_pair_counts(test_lp, train_lp)
print(f'plates: test {len(test_lp)}, train {len(train_lp)}; equal-length pairs compared: {sum(cnt.values()):,}')
print('k = positions where both plates are visible; match = all k agree; near = exactly one of the k differs')
print(f'{"k":>3} {"pairs":>10} {"match":>7} {"near":>7} {"match rate":>11}')
for k in sorted({k for k, _ in cnt}):
    n = sum(v for (kk, _), v in cnt.items() if kk == k)
    m, nr = cnt.get((k, k), 0), cnt.get((k, k - 1), 0)
    print(f'{k:>3} {n:>10,} {m:>7} {nr:>7} {m / n:>11.2e}')

plates: test 491, train 2056; equal-length pairs compared: 994,113
k = positions where both plates are visible; match = all k agree; near = exactly one of the k differs
  k      pairs   match    near  match rate
  3    400,796     243    7665    6.06e-04
  4    470,704      27    1017    5.74e-05
  5    122,610     176      28    1.44e-03
  6          3       0       0    0.00e+00


**Stop here.** Report the timings and the table to the author; do not start any long run before the reply (PLAN.md, M1b-7).

# Step 2: plate-text pairs, cheap checks, controls, ORB for checks 3 and 1
Run top to bottom after the cells above (the setup, helpers and parameters cells must have run in this session). Everything is cached on Drive (`reports/leak_check`), so after a disconnect re-run from the top and it continues.

**Output rules:** counts and timings only. No plate text, no file names, no images.
**Nothing is removed in this notebook.** The ORB cells only count and apply the 5% guard.
**Long cells** (the two controls and the two ORB checks) run only when their flag in the next cell is `True`. Each prints its cost first; read it, then set the flag. Never run a control and an ORB check in the same cell: they are separate cells.
**Check 4 (CCPD held-out vs CCPD train) has no ORB cell on purpose** (about 5.5 h, and there is no copy-free same-domain control).

In [ ]:
# Parameters (step 2)
import json
OUT = Path(DRIVE, 'reports/leak_check')          # caches of this notebook (private Drive folder, never the repo)
(OUT / 'desc').mkdir(parents=True, exist_ok=True)
DHASH_MAX = 6          # Hamming <= this = match (PLAN.md rule)
T_FLOOR = 42           # ORB threshold floor (M1b-3/4); T = max(T_FLOOR, highest control score + 1)
KEEP = 20              # controls and ORB checks store every pair with at least this many inliers
N_CONTROL = 300        # UC3M images in the positive control
GUARD = 0.05           # a rule that would remove more than this share of a training set trips the guard
BANDS = (20, 30, 42, 60, 100)   # score bands printed after each ORB check (lower edges; the last band is open)
SHARD = 250            # images per descriptor cache file
UC3M_SIZE_OVERRIDE = None       # None = the size picked by the positive control; or 1024 / 1536
RUN_CTRL1 = False      # negative control for check 1: US test x CCPD validation
RUN_CTRL3 = False      # negative control for check 3: UC3M test x CCPD validation
RUN_ORB3 = False       # ORB check 3: UC3M test x UC3M train
RUN_ORB1 = False       # ORB check 1: US test x CCPD train

In [ ]:
# helpers (step 2)
from alpr import roboflow as rf
def keys_of(name, split): return sorted(f'{name}/{m}' for m in MEM[name][split])   # key = tar name + '/' + member
def read_key(k): n, m = k.split('/', 1); return read(n, m)

_last = {}
def progress(label, i, n, t0, every=30.0):
    now = time.time()
    if now - _last.get(label, 0.0) >= every or i == n:
        _last[label] = now
        print(f'  {label}: {i}/{n}, {now - t0:.0f} s elapsed', flush=True)

def get_desc(name, split, size):
    """ORB descriptors of one tar split at one working size, cached on Drive in shards of SHARD images (resumes after a disconnect)."""
    ks, out, bad, t0 = keys_of(name, split), {}, 0, time.time()
    for s in range(0, len(ks), SHARD):
        f = OUT / 'desc' / f'{name}_{split}_{size or "native"}_f{ORB_FEATURES}_s{SHARD}_{s // SHARD:04d}.pkl'
        if f.exists():
            with open(f, 'rb') as fh:
                out.update(pickle.load(fh))   # our own cache file
        else:
            d = {}
            for k in ks[s:s + SHARD]:
                try:
                    d[k] = describe(load_gray(read_key(k), size))
                except (OSError, ValueError):
                    d[k] = (np.zeros((0, 2), np.float32), None); bad += 1
            with open(f.with_suffix('.tmp'), 'wb') as fh:
                pickle.dump(d, fh)
            f.with_suffix('.tmp').rename(f); out.update(d)
        progress(f'descriptors {name}/{split}', min(s + SHARD, len(ks)), len(ks), t0)
    if bad: print(f'  {name}/{split}: {bad} unreadable images (no descriptors, they score 0)')
    return out

# timings from the cells above are saved to Drive, so a restarted kernel still knows the RANSAC cost
TJ = OUT / 'timing.json'
if 'PAIR' in globals():
    json.dump({k: [float(x) if x is not None else None for x in v] for k, v in PAIR.items()}, open(TJ, 'w'))
_rs = [v[2] for v in json.load(open(TJ)).values() if v[2] is not None] if TJ.exists() else []
RANSAC_MEAN = float(np.mean(_rs)) if _rs else None
print('RANSAC cost per call from the timing step:', f'{RANSAC_MEAN * 1000:.2f} ms' if RANSAC_MEAN else 'not available (cost below then excludes RANSAC)')

def quick_time(DESX, Q, Tg, n=200):
    """Seconds per pair on n random pairs of the loaded descriptors (cost only): knn+ratio, plus the RANSAC share x the mean RANSAC call."""
    r = random.Random(SEED + 5); res = [score_timed(DESX[r.choice(Q)], DESX[r.choice(Tg)]) for _ in range(n)]
    return np.mean([x[1] for x in res]) + np.mean([x[2] is not None for x in res]) * (RANSAC_MEAN or 0)

def cost_line(label, DESX, Q, Tg):
    per = quick_time(DESX, Q, Tg); n = len(Q) * len(Tg)
    print(f'COST {label}: {len(Q)} x {len(Tg)} = {n:,} pairs x {per * 1000:.2f} ms = {n * per / 3600:.2f} h (single process; random-pair timing; RANSAC on real matches adds a little)')

def control_T(hits, done, Q, Tg):
    """(T, highest stored score or None) once the control block is complete, else (None, None). T = max(T_FLOOR, highest + 1)."""
    if rf.pending(Q, Tg, done): return None, None
    top = max(rf.read_hits(hits).values(), default=None)
    return (max(T_FLOOR, top + 1) if top is not None else T_FLOOR), top

def orb_keys(a, b): return score_timed(DES[a], DES[b])[0]

In [ ]:
# 1. Plate text, check 3: who is behind the k = 5 and k = 4 matches (counts only)
rows = list(csv.DictReader(open(LABELS_CSV, newline='', encoding='utf-8')))
te = [(r['image_id'], r['lp_id']) for r in rows if r['split'] == 'test']
tr = [(r['image_id'], r['lp_id']) for r in rows if r['split'] == 'train']
cnt = uc3m.plate_pair_counts([l for _, l in te], [l for _, l in tr])
TEXT = {}   # k -> (test image ids, train image ids)
for k in (5, 4):
    m = uc3m.plate_matches([l for _, l in te], [l for _, l in tr], k)
    ti, ri = {te[i][0] for i, _, _ in m}, {tr[j][0] for _, j, _ in m}
    TEXT[k] = (ti, ri)
    print(f'k={k}: {len(m)} matching plate pairs (table says {cnt.get((k, k), 0)}: {"agree" if len(m) == cnt.get((k, k), 0) else "DIFFER"}) | '
          f'test plates {len({i for i, _, _ in m})}, train plates {len({j for _, j, _ in m})} | '
          f'distinct test images {len(ti)} of {len({i for i, _ in te})}, distinct train images {len(ri)} of {len({i for i, _ in tr})} | '
          f'pairs with identical masked positions {sum(s for _, _, s in m)} of {len(m)}')
json.dump({str(k): [sorted(v[0]), sorted(v[1])] for k, v in TEXT.items()}, open(OUT / 'plate_text_images.json', 'w'))   # image ids only

In [ ]:
# 2. Cheap checks: MD5 and dHash on all pairs (hashes cached on Drive; no ORB)
SETS = {'us_test': [('us', 'test')], 'us_pool': [('us', 'train'), ('us', 'val')], 'ccpd_train': [('ccpd', 'train')], 'ccpd_held': [('ccpd', 'heldout')],
        'uc3m_all': [('uc3m', 'train'), ('uc3m', 'test')], 'uc3m_test': [('uc3m', 'test')], 'uc3m_train': [('uc3m', 'train')]}
S = {n: sorted(k for a, b in v for k in keys_of(a, b)) for n, v in SETS.items()}
CHEAP = {'1': ('us_test', 'ccpd_train'), '2a': ('uc3m_all', 'ccpd_train'), '2b': ('uc3m_all', 'us_pool'), '3': ('uc3m_test', 'uc3m_train'), '4': ('ccpd_held', 'ccpd_train'),
         'x1 CCPD held-out vs US pool': ('ccpd_held', 'us_pool'), 'x2 CCPD held-out vs UC3M train': ('ccpd_held', 'uc3m_train'), 'x3 US test vs UC3M train': ('us_test', 'uc3m_train')}
def dhash64(g):   # same hash as notebooks 03 and 04: 9x8 LANCZOS, left > right
    px = np.asarray(Image.fromarray(g).resize((9, 8), Image.LANCZOS), dtype=np.int16)
    return int(''.join('1' if b else '0' for b in (px[:, 1:] > px[:, :-1]).ravel()), 2)

H_CSV = OUT / 'hashes.csv'
HASH = {r['key']: (r['md5'], r['dhash']) for r in csv.DictReader(open(H_CSV, newline=''))} if H_CSV.exists() else {}
todo = [k for v in S.values() for k in v if k not in HASH]; todo = sorted(set(todo))
print(f'{len(HASH)} hashes cached, {len(todo)} to do (UC3M dHash is taken from a draft decode at long side 1024: the 9x8 hash does not need more)')
t0, new = time.time(), not H_CSV.exists()
with open(H_CSV, 'a', newline='') as f:
    wr = csv.writer(f)
    if new: wr.writerow(['key', 'md5', 'dhash'])
    for i, k in enumerate(todo, 1):
        data = read_key(k)
        try: h = format(dhash64(load_gray(data, 1024 if k.startswith('uc3m/') else None)), '016x')
        except (OSError, ValueError): h = 'bad'
        HASH[k] = (hashlib.md5(data).hexdigest(), h); wr.writerow([k, *HASH[k]])
        if i % 200 == 0: f.flush()
        progress('hash', i, len(todo), t0)
print('unreadable images (no dHash):', sum(v[1] == 'bad' for v in HASH.values()))

LUT = np.array([bin(i).count('1') for i in range(256)], np.uint8)
def dh_hits(qs, ts):
    tk = [k for k in ts if HASH[k][1] != 'bad']; th = np.array([int(HASH[k][1], 16) for k in tk], np.uint64); out = []
    for q in qs:
        if HASH[q][1] == 'bad': continue
        d = LUT[(th ^ np.uint64(int(HASH[q][1], 16))).view(np.uint8)].reshape(-1, 8).sum(1)
        out += [(q, tk[j], int(d[j])) for j in np.flatnonzero(d <= DHASH_MAX)]
    return out
def md5_hits(qs, ts):
    by = {}
    for t in ts: by.setdefault(HASH[t][0], []).append(t)
    return [(q, t) for q in qs for t in by.get(HASH[q][0], [])]

CHEAP_HITS = {}
print(f'{"check":<32}{"pairs":>12} | {"MD5 pairs":>9} {"(q imgs/t imgs)":>16} | {"dHash pairs":>11} {"(q imgs/t imgs)":>16} | train images flagged / size | guard')
for name, (qn, tn) in CHEAP.items():
    qs, ts = S[qn], S[tn]; m, d = md5_hits(qs, ts), dh_hits(qs, ts)
    CHEAP_HITS[name] = dict(md5=m, dhash=d)
    fl = {t for _, t in m} | {t for _, t, _ in d}
    print(f'{name:<32}{len(qs) * len(ts):>12,} | {len(m):>9} {f"({len({q for q, _ in m})}/{len({t for _, t in m})})":>16} | {len(d):>11} {f"({len({q for q, _, _ in d})}/{len({t for _, t, _ in d})})":>16} | '
          f'{len(fl)} / {len(ts)} = {len(fl) / len(ts):.2%} | {"TRIPPED" if len(fl) / len(ts) > GUARD else "pass"}')
json.dump(CHEAP_HITS, open(OUT / 'cheap_hits.json', 'w'))   # file keys only, Drive only
print('Share = distinct training-side images flagged by MD5 or dHash, without name-copy expansion. Nothing is removed here.')

In [ ]:
# 3. Positive control for check 3: 300 altered UC3M test images vs their originals, at both working sizes
ctrl = random.Random(SEED + 11).sample(keys_of('uc3m', 'test'), N_CONTROL)
POS = {}
for size in UC3M_SIZES:
    f = OUT / f'poscontrol_{size}.json'
    if f.exists():
        POS[size] = json.load(open(f)); continue
    r, res, t0 = random.Random(SEED + size), [], time.time()
    for i, k in enumerate(ctrl, 1):
        g = load_gray(read_key(k), size)
        res.append(score_timed(describe(g), describe(altered(g, r)))[0])   # alteration is applied after the resize (an approximation of altering the full-size file)
        progress(f'positive control {size}', i, len(ctrl), t0)
    json.dump(res, open(f, 'w')); POS[size] = res
for size, v in POS.items():
    a = np.array(v)
    print(f'size {size}: recall at T={T_FLOOR}: {(a >= T_FLOOR).mean():.3f} ({int((a >= T_FLOOR).sum())}/{len(a)}) | lowest inliers {a.min()} | 5th percentile {np.percentile(a, 5):.0f} | median {np.median(a):.0f}')
PICK = max(UC3M_SIZES, key=lambda s: (np.mean(np.array(POS[s]) >= T_FLOOR), -s))   # higher recall wins; a tie goes to the smaller (cheaper) size
UC3M_SIZE = UC3M_SIZE_OVERRIDE or PICK
print(f'PICK: working size {PICK} (rule: higher recall at T={T_FLOOR}, tie -> smaller size)' + (f'; OVERRIDDEN to {UC3M_SIZE}' if UC3M_SIZE_OVERRIDE else ''))

In [ ]:
# 4a. descriptors for the controls and ORB checks (cached in shards on Drive; resumes)
DES = {}
for name, split, size in [('ccpd', 'val', None), ('us', 'test', None), ('uc3m', 'test', UC3M_SIZE), ('uc3m', 'train', UC3M_SIZE), ('ccpd', 'train', None)]:
    DES.update(get_desc(name, split, size))
print(len(DES), 'descriptor sets loaded')

In [ ]:
# 4b. negative control for check 1: US test x CCPD validation (complete block; stores every pair >= KEEP)
Q, Tg = keys_of('us', 'test'), keys_of('ccpd', 'val')
H_C1, D_C1 = OUT / 'ctrl1_us_ccpdval.csv', OUT / 'ctrl1_us_ccpdval.done.csv'
cost_line('control 1', DES, Q, Tg)
if RUN_CTRL1: rf.search_pairs(Q, Tg, orb_keys, H_C1, D_C1, KEEP, 'control 1')
else: print('Skipped: RUN_CTRL1 is False. Read the cost, then set it in the step-2 parameters cell.')
T1, top1 = control_T(H_C1, D_C1, Q, Tg)
print('control 1 not finished' if T1 is None else f'control 1: highest stored score {top1 if top1 is not None else f"< {KEEP}"} -> T1 = max({T_FLOOR}, highest + 1) = {T1}')

In [ ]:
# 4c. negative control for check 3: UC3M test x CCPD validation (complete block, UC3M at the picked size)
Q, Tg = keys_of('uc3m', 'test'), keys_of('ccpd', 'val')
H_C3, D_C3 = OUT / f'ctrl3_uc3m_ccpdval_{UC3M_SIZE}.csv', OUT / f'ctrl3_uc3m_ccpdval_{UC3M_SIZE}.done.csv'
cost_line('control 3', DES, Q, Tg)
if RUN_CTRL3: rf.search_pairs(Q, Tg, orb_keys, H_C3, D_C3, KEEP, 'control 3')
else: print('Skipped: RUN_CTRL3 is False. Read the cost, then set it in the step-2 parameters cell.')
T3, top3 = control_T(H_C3, D_C3, Q, Tg)
print('control 3 not finished' if T3 is None else f'control 3: highest stored score {top3 if top3 is not None else f"< {KEEP}"} -> T3 = max({T_FLOOR}, highest + 1) = {T3}')
print('Note: this control is cross-domain (Spanish vs Chinese images); same-scene chance matches inside UC3M are not covered by it.')

In [ ]:
# report helper for the ORB checks: counts by score band and the 5% guard (nothing is removed)
def orb_report(hits_csv, T, cheap_name, n_train, extra=()):
    h = rf.read_hits(hits_csv); edges = list(BANDS) + [10**9]
    print(f'pairs stored (score >= {KEEP}): {len(h)}')
    for lo, hi in zip(edges, edges[1:]):
        sel = [(q, t) for (q, t), s in h.items() if lo <= s < hi]
        print(f'  score {lo}-{hi - 1 if hi < 10**9 else "up"}: {len(sel)} pairs, {len({q for q, _ in sel})} test images, {len({t for _, t in sel})} train images')
    hit = [(q, t) for (q, t), s in h.items() if s >= T]
    orb_t = {t for _, t in hit}
    ch = json.load(open(OUT / 'cheap_hits.json'))[cheap_name]
    cheap_t = {t for _, t in ch['md5']} | {t for _, t, _ in ch['dhash']}
    rows = [('ORB >= T only', orb_t), ('ORB >= T + MD5 + dHash', orb_t | cheap_t)] + [(l, orb_t | cheap_t | fl) for l, fl in extra]
    for label, fl in rows:
        print(f'  T = {T}: {label}: {len(fl)} of {n_train} training images = {len(fl) / n_train:.2%} -> 5% guard {"TRIPPED (stop, remove nothing)" if len(fl) / n_train > GUARD else "pass"}')
    print('  (distinct training-side images, no name-copy expansion; nothing removed)')

In [ ]:
# 5a. ORB check 3: UC3M test x UC3M train, at the picked size (own cell; resumes)
Q, Tg = keys_of('uc3m', 'test'), keys_of('uc3m', 'train')
H_O3, D_O3 = OUT / f'orb3_{UC3M_SIZE}.csv', OUT / f'orb3_{UC3M_SIZE}.done.csv'
cost_line('ORB check 3', DES, Q, Tg)
if T3 is None: print('Control 3 is not finished, so T is unknown: run the controls first.')
elif RUN_ORB3: rf.search_pairs(Q, Tg, orb_keys, H_O3, D_O3, KEEP, 'ORB check 3')
else: print('Skipped: RUN_ORB3 is False. Read the cost, then set it in the step-2 parameters cell.')
if T3 is not None and not rf.pending(Q, Tg, D_O3):
    stem = {Path(k).stem: k for k in Tg}
    ex = [(f'+ plate text k=5 ({len(TEXT[5][1])} train images)', {stem[i] for i in TEXT[5][1] if i in stem})] if 'TEXT' in globals() else []
    orb_report(H_O3, T3, '3', len(Tg), ex)
elif T3 is not None: print(f'{len(rf.pending(Q, Tg, D_O3))} of {len(Q)} queries still to search.')

In [ ]:
# 5b. ORB check 1: US test x CCPD train (own cell; resumes)
Q, Tg = keys_of('us', 'test'), keys_of('ccpd', 'train')
H_O1, D_O1 = OUT / 'orb1.csv', OUT / 'orb1.done.csv'
cost_line('ORB check 1', DES, Q, Tg)
if T1 is None: print('Control 1 is not finished, so T is unknown: run the controls first.')
elif RUN_ORB1: rf.search_pairs(Q, Tg, orb_keys, H_O1, D_O1, KEEP, 'ORB check 1')
else: print('Skipped: RUN_ORB1 is False. Read the cost, then set it in the step-2 parameters cell.')
if T1 is not None and not rf.pending(Q, Tg, D_O1): orb_report(H_O1, T1, '1', len(Tg))
elif T1 is not None: print(f'{len(rf.pending(Q, Tg, D_O1))} of {len(Q)} queries still to search.')

**Stop here.** Report the plate-text counts, the cheap-check table, the positive-control recall and the control costs. No ORB check starts until the author replies. Check 4 ORB is skipped on purpose; PLAN.md known limits are updated after the author confirms.